تحميل المكتبات المطلوبة

In [ ]:
import numpy
import sys
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords
from keras.models import Sequential
from keras.layers import Dense, Dropout, LSTM
from keras.utils import to_categorical
from keras.callbacks import ModelCheckpoint

import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
from nltk.tokenize import RegexpTokenizer

# Define pattern: Match sequences of word characters
tokenizer = RegexpTokenizer(r'\w+',gaps=False)

text = "Hello, world! Welcome to NLTK (v3.8) tokenization."
tokens = tokenizer.tokenize(text)

print(tokens)

['Hello', 'world', 'Welcome', 'to', 'NLTK', 'v3', '8', 'tokenization']


In [ ]:
from nltk.tokenize import RegexpTokenizer

# Pattern matches prices like $19.99 or standalone numbers/words
pattern = r'\$?\d+(?:\.\d+)?|\w+'
tokenizer = RegexpTokenizer(pattern)

text = "The price of the item is $19.99, but with tax it is 21.50 dollars."
tokens = tokenizer.tokenize(text)

print(tokens)

['The', 'price', 'of', 'the', 'item', 'is', '$19.99', 'but', 'with', 'tax', 'it', 'is', '21.50', 'dollars']


In [ ]:
from nltk.tokenize import RegexpTokenizer

# Treat whitespace as the gap/delimiter to split on
tokenizer = RegexpTokenizer(r'\s+', gaps=True)

text = "Data Science, Machine Learning & AI!"
tokens = tokenizer.tokenize(text)

print(tokens)

['Data', 'Science,', 'Machine', 'Learning', '&', 'AI!']


In [ ]:
import numpy as np
from keras.utils import to_categorical

# Integer class labels (e.g., 3 classes: 0, 1, 2)
labels = np.array([0, 1, 2, 1, 0])

# Perform one-hot encoding
encoded = to_categorical(labels)

print("Original Labels Shape:", labels.shape)
print("Encoded Matrix Shape: ", encoded.shape)
print("\nEncoded Output:\n", encoded)

Original Labels Shape: (5,)
Encoded Matrix Shape:  (5, 3)

Encoded Output:
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [0. 1. 0.]
 [1. 0. 0.]]


In [ ]:
import numpy as np
from keras.utils import to_categorical

# Labels with max class index = 2
labels = np.array([0, 2, 1])

# Specify total classes = 5 (indices 0 through 4)
encoded = to_categorical(labels, num_classes=5)

print("Encoded Matrix Shape:", encoded.shape)
print("\nEncoded Output:\n", encoded)

Encoded Matrix Shape: (3, 5)

Encoded Output:
 [[1. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 1. 0. 0. 0.]]


قراءة الملفات

In [ ]:
with open("Franknestein.txt", encoding="utf8") as text_file:
    file = text_file.read()
# print(file)

عدد دالة لتناول الكلمات و حذف كلمات التوقف

In [ ]:
def tokenize_words(input):
    # lowercase everything to standardize it
    input = input.lower()

    # instantiate the tokenizer
    tokenizer = RegexpTokenizer(r'\w+')
    tokens = tokenizer.tokenize(input)

    # if the created token isn't in the stop words, make it part of "filtered"
    filtered = filter(lambda token: token not in stopwords.words('english'), tokens)
    return " ".join(filtered)

In [ ]:
processed_inputs = tokenize_words(file)

عدد الحروف

In [ ]:
len(processed_inputs)

269995

اول الف حرف

In [ ]:
processed_inputs[:1000]

'project gutenberg frankenstein mary wollstonecraft godwin shelley ebook use anyone anywhere cost almost restrictions whatsoever may copy give away use terms project gutenberg license included ebook online www gutenberg net title frankenstein modern prometheus author mary wollstonecraft godwin shelley release date june 17 2008 ebook 84 last updated january 13 2018 language english character set encoding utf 8 start project gutenberg ebook frankenstein produced judith boss christy phillips lynn hanninen david meltzer html version al haines corrections menno de leeuw frankenstein modern prometheus mary wollstonecraft godwin shelley contents letter 1 letter 2 letter 3 letter 4 chapter 1 chapter 2 chapter 3 chapter 4 chapter 5 chapter 6 chapter 7 chapter 8 chapter 9 chapter 10 chapter 11 chapter 12 chapter 13 chapter 14 chapter 15 chapter 16 chapter 17 chapter 18 chapter 19 chapter 20 chapter 21 chapter 22 chapter 23 chapter 24 letter 1 _to mrs saville england _ st petersburgh dec 11th 17 

حذف التكرار و ترتيبها

In [ ]:
chars = sorted(list(set(processed_inputs)))

In [ ]:
len(chars)

43

اول 15 حرف

In [ ]:
chars[:15]

[' ', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '_', 'a', 'b', 'c']

عمل قاموس للتحويل من حرف لرقم

In [ ]:
char_to_num = dict((c, i) for i, c in enumerate(chars))

In [ ]:
char_to_num

{' ': 0,
 '0': 1,
 '1': 2,
 '2': 3,
 '3': 4,
 '4': 5,
 '5': 6,
 '6': 7,
 '7': 8,
 '8': 9,
 '9': 10,
 '_': 11,
 'a': 12,
 'b': 13,
 'c': 14,
 'd': 15,
 'e': 16,
 'f': 17,
 'g': 18,
 'h': 19,
 'i': 20,
 'j': 21,
 'k': 22,
 'l': 23,
 'm': 24,
 'n': 25,
 'o': 26,
 'p': 27,
 'q': 28,
 'r': 29,
 's': 30,
 't': 31,
 'u': 32,
 'v': 33,
 'w': 34,
 'x': 35,
 'y': 36,
 'z': 37,
 'æ': 38,
 'è': 39,
 'é': 40,
 'ê': 41,
 'ô': 42}

عدد الحروف في النص و عددها دون تكرار

In [ ]:
input_len = len(processed_inputs)
vocab_len = len(chars)
print ("Total number of characters:", input_len)
print ("Total vocab:", vocab_len)

Total number of characters: 269995
Total vocab: 43


اعداد المعاملات

In [ ]:
seq_length = 100
x_data = []
y_data = []

تقطيع النص

In [ ]:
# loop through inputs, start at the beginning and go until we hit
# the final character we can create a sequence out of
for i in range(0, input_len - seq_length, 1):
    # Define input and output sequences
    # Input is the current character plus desired sequence length
    in_seq = processed_inputs[i:i + seq_length]

    # Out sequence is the initial character plus total sequence length
    out_seq = processed_inputs[i + seq_length]

    # We now convert list of characters to integers based on
    # previously and add the values to our lists
    x_data.append([char_to_num[char] for char in in_seq])
    y_data.append(char_to_num[out_seq])

عدد الحروف

In [ ]:
n_patterns = len(x_data)
print ("Total Patterns:", n_patterns)

Total Patterns: 269895


غيير الابعاد

In [ ]:
X = numpy.reshape(x_data, (n_patterns, seq_length, 1))
X = X/float(vocab_len)

عمل

One hot encoder

 للشبكة

In [ ]:
y =to_categorical(y_data)

الشبكة

In [ ]:
model = Sequential()
model.add(LSTM(256, input_shape=(X.shape[1], X.shape[2]), return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(256, return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(128))
model.add(Dropout(0.2))
model.add(Dense(y.shape[1], activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
model.compile(loss='categorical_crossentropy', optimizer='adam')

تحديد ملف لحفظ الاوزان

In [ ]:
filepath = "model_weights_saved.keras"
checkpoint = ModelCheckpoint(filepath, monitor='loss', verbose=1, save_best_only=True, mode='min')
desired_callbacks = [checkpoint]

التدريب

In [ ]:
model.fit(X, y, epochs=20, batch_size=256, callbacks=desired_callbacks)

Epoch 1/20
1054/1055 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - loss: 2.9684
Epoch 1: loss improved from None to 2.91909, saving model to model_weights_saved.keras

Epoch 1: finished saving model to model_weights_saved.keras
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 76s 67ms/step - loss: 2.9191
Epoch 2/20
1054/1055 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - loss: 2.6876
Epoch 2: loss improved from 2.91909 to 2.64468, saving model to model_weights_saved.keras

Epoch 2: finished saving model to model_weights_saved.keras
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 74s 70ms/step - loss: 2.6447
Epoch 3/20
1054/1055 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - loss: 2.5221
Epoch 3: loss improved from 2.64468 to 2.48730, saving model to model_weights_saved.keras

Epoch 3: finished saving model to model_weights_saved.keras
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 73s 69ms/step - loss: 2.4873
Epoch 4/20
1054/1055 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - loss: 2.3835
Epoch 4: loss improved from 2.48730 to 2.35885, saving model to model_weights_saved.keras


فتح الملف المحفوظ , وتحميل الاوزان

In [ ]:
filename = "model_weights_saved.keras"
model.load_weights(filename)
model.compile(loss='categorical_crossentropy', optimizer='adam')

عمل القاموس

In [ ]:
num_to_char = dict((i, c) for i, c in enumerate(chars))

عمل رقم عشوائي لاختيار 100 حرف من النص لتبدا به الشبكة

In [ ]:
start = numpy.random.randint(0, len(x_data) - 1)
pattern = x_data[start]
print("Random Seed:")
print("\"", ''.join([num_to_char[value] for value in pattern]), "\"")

Random Seed:
" ring summer passed away labours watch blossom expanding leaves sights always yielded supreme delight "


التوقع

In [ ]:
for i in range(1000):
    x = numpy.reshape(pattern, (1, len(pattern), 1))
    x = x / float(vocab_len)
    prediction = model.predict(x, verbose=0)
    index = numpy.argmax(prediction)
    result = num_to_char[index]
    seq_in = [num_to_char[value] for value in pattern]

    sys.stdout.write(result)

    pattern.append(index)
    pattern = pattern[1:len(pattern)]

 searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searon searo